In [10]:
import os
import dotenv
import json
from openai import OpenAI
from IPython.display import display, Markdown, update_display
from scraper import fetch_website_links, fetch_website_contents
import requests


In [2]:
openai = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'
)

In [4]:
links = fetch_website_links("https://gnani.ai")
links 

['https://vachana.ai/',
 '/',
 '/inya-workforce-automate365-ai',
 '/inya-assist-assist365-agent-assist-ai',
 '/inya-shield-armour365-voice-biometrics-ai',
 '/inya-insights-aura365-conversation-analytics-ai',
 'https://inya.ai/',
 '/marketing-automation-ai',
 '/customer-experience-ai',
 '/customer-support-ai',
 '/debt-collections-voice-ai',
 '/agent-assist-ai',
 '/analytics-and-qa-ai',
 '/internal-operations-automation',
 '/agentic-ai-for-banking',
 '/agentic-ai-for-insurance',
 '/agentic-ai-for-automotive',
 '/agentic-ai-for-consumer-durables',
 '/agentic-ai-for-bpo-contact-centers',
 '/agentic-ai-for-telecom',
 '/agentic-ai-for-real-estate',
 '/agentic-ai-for-healthcare',
 '/agentic-ai-for-hospitality',
 '/agentic-ai-for-edtech',
 '/technology#asr',
 '/technology#tts',
 '/technology#translate',
 '/technology#slm-rags',
 '/technology#language-switch-identification',
 '/technology#barge-handling',
 '/technology#noice-cancellation',
 '/technology#accent-change',
 '/resources/research',
 

In [5]:
links_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://openai.com/about"},
        {"type": "careers page", "url": "https://openai.com/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://gnani.ai"))


Here is the list of links on the website https://gnani.ai -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://vachana.ai/
/
/inya-workforce-automate365-ai
/inya-assist-assist365-agent-assist-ai
/inya-shield-armour365-voice-biometrics-ai
/inya-insights-aura365-conversation-analytics-ai
https://inya.ai/
/marketing-automation-ai
/customer-experience-ai
/customer-support-ai
/debt-collections-voice-ai
/agent-assist-ai
/analytics-and-qa-ai
/internal-operations-automation
/agentic-ai-for-banking
/agentic-ai-for-insurance
/agentic-ai-for-automotive
/agentic-ai-for-consumer-durables
/agentic-ai-for-bpo-contact-centers
/agentic-ai-for-telecom
/agentic-ai-for-real-estate
/agentic-ai-for-healthcare
/agentic-ai-for-hospitality
/agentic-ai-for-edtech
/technology#asr
/technology#tts
/technology#translate
/technolog

In [12]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="qwen2.5:latest",
        messages=[
            {'role': 'system', 'content': links_system_prompt},
            {'role': 'user', 'content': get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://gnani.ai")

Found 2 relevant links


{'links': [{'type': 'about page', 'url': 'https://inya.ai/about-us'},
  {'type': 'careers page', 'url': 'https://inya.ai/careers'}]}

In [15]:
select_relevant_links("https://edwarddonner.com")

Found 3 relevant links


{'links': [{'type': 'about me page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'posts blog articles', 'url': 'https://edwarddonner.com/posts/'}]}

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url) #Fetch homepage contents
    relevant_links = select_relevant_links(url) #Select relevant links from homepage
    result = f" ## Landing page content for {url} \n\n {contents} \n\n"
    for link in relevant_links['links']:
        result += f'\n\n ## Conetent for {link['type']}'
        result += fetch_website_contents(link["url"])
    return result


In [ ]:
print(fetch_page_and_all_relevant_links("https://openai.com"))

In [23]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [24]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
print(get_brochure_user_prompt("OpenAI", "https://openai.com"))

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="qwen2.5:latest",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or '' #“Take the new chunk and add it to previous response.”
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found 3 relevant links


# Hugging Face - The AI Community Building the Future

## Introduction
Hugging Face is a platform that fosters collaboration among machine learning (ML) enthusiasts, researchers, and developers. We provide a vibrant community and powerful tools to explore and build ML models, datasets, applications, and more. Our mission is to drive innovation in artificial intelligence.

---

### **Explore Models**
With over 2 million public ML models at your fingertips, Hugging Face is the go-to place for discovering and leveraging cutting-edge AI models across various domains, from Natural Language Processing (NLP) to Computer Vision (CV).

#### Popular Models:
- **Sulphur-2-base** by SulphurAI: Updated just 1 day ago
- **MiniCPM-V-4.6** by openbmb: Updated about 23 hours ago
- **Supertonic-3** by Supertone: Updated 1 day ago

#### Notable Spaces:
- **Pixal3D**: High-fidelity pixel-aligned image-to-3D generation
- **Supertonic 3 (TTS)**: Fast, on-device multilingual TTS
- **LTX 2.3 Studio**: Generate videos from any input including text and images

---

### **Collaborate with Datasets**
Explore our vast library of datasets to train your models effectively. With over 500K datasets available and growing, you can find or contribute to the latest research in NLP, computer vision, audio, and more.

#### Featured Datasets:
- **SynData** by PsiBotAI: Updated about 9 hours ago
- **Open-MM-RL** by TuringEnterprises: Updated 6 days ago

---

### **Join Our Community**
Hugging Face is not just a platform; it’s a community where knowledge, ideas, and innovations thrive. Connect with other ML enthusiasts through our forums, Discord channels, and events.

#### Social Channels:
- **Discord**: Join the Hugging Face Discord for real-time discussions
- **Blog**: Visit our blog for articles on the latest trends in AI

---

### **Careers and Opportunities**
Join Hugging Face and be part of a dynamic team driving advancements in AI. From software engineers to data scientists, we have roles that align with your skills and passion.

#### Current Openings:
Explore our [careers page](https://huggingface.co/careers) for the latest job opportunities.

---

### **Investment and Enterprise Support**
For companies looking to leverage AI, Hugging Face provides enterprise solutions including inference endpoints, storage buckets, and custom support tailored to your needs.

#### Contact Us:
- **Website**: Visit our website at [huggingface.co](https://huggingface.co)
- **Support**: Reach out for further details on enterprise services

---

### **Conclusion**
At Hugging Face, we're committed to making AI accessible to everyone. Our community-driven approach and powerful tools ensure you stay ahead in the ever-evolving field of artificial intelligence.

Join us and be part of the future of machine learning!

--- 

For more information, visit [huggingface.co](https://huggingface.co/) and explore the possibilities of tomorrow.